# Qwen-14B Map Reasoning Probe

This notebook tests whether `Qwen/Qwen2.5-14B-Instruct` can read a compact LPLH2-style world map and answer route questions.

It does not run Zork. It only loads Qwen-14B and asks map-comprehension questions against a static discovered Zork1 map.

The test includes:
- known route questions,
- object-location route questions,
- unknown route/object questions where the correct answer is to say the route/object is not known from the KG.


In [ ]:
!pip -q install -U transformers accelerate sentencepiece


In [ ]:
import json, re, textwrap, collections
from collections import deque
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
MAX_NEW_TOKENS = 256

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("CUDA available:", torch.cuda.is_available())


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Loaded", MODEL_NAME, "with dtype", dtype)


## Static World Model

This uses the simpler JSON shape we discussed:
- current location,
- current room state,
- navigation graph,
- routes from current location,
- frontier.

I added `known_object_locations` only for object-route questions. If an object is not in that field, the model should say it is unknown rather than using outside Zork knowledge.

In [ ]:
WORLD_MODEL = {
    "current_location": "Living Room",
    "current_room_state": {
        "inventory": ["sword", "lantern", "sack", "bottle"],
        "visible_objects": ["trophy case", "rug", "trap door"],
        "object_states": {
            "trophy case": "open",
            "rug": "moved",
            "trap door": "revealed, open"
        },
        "confirmed_exits": {
            "east": "Kitchen",
            "down": "Cellar"
        },
        "blocked_exits": {
            "west": "nailed wooden door"
        },
        "untried_exits": ["north", "south"]
    },
    "navigation_graph": {
        "West of House": {"north": "North of House", "west": "Forest"},
        "North of House": {"south": "West of House", "north": "Forest Path", "east": "Behind House"},
        "Forest Path": {"south": "North of House", "north": "Clearing", "up": "Up a Tree", "east": "Forest", "west": "Forest"},
        "Up a Tree": {"down": "Forest Path"},
        "Clearing": {"south": "Forest Path"},
        "Forest": {"east": "Forest Path"},
        "Behind House": {"west": "North of House", "east": "Kitchen"},
        "Kitchen": {"west": "Living Room", "east": "Behind House", "up": "Attic"},
        "Attic": {"down": "Kitchen"},
        "Living Room": {"east": "Kitchen", "down": "Cellar"},
        "Cellar": {"up": "Living Room", "north": "Troll Room", "south": "East-West Passage"},
        "Troll Room": {"south": "Cellar", "east": "East-West Passage"},
        "East-West Passage": {"west": "Troll Room", "east": "Loud Room", "north": "Round Room"},
        "Loud Room": {"west": "East-West Passage"},
        "Round Room": {"south": "East-West Passage"}
    },
    "known_object_locations": {
        "egg": "Up a Tree",
        "lantern": "Living Room",
        "sword": "Living Room",
        "sack": "Kitchen",
        "bottle": "Kitchen",
        "axe": "Troll Room"
    },
    "routes_from_current_location": {},
    "frontier": {
        "Clearing": ["east", "west", "north"],
        "Cellar": ["west", "down"],
        "East-West Passage": ["down"],
        "Loud Room": ["up"]
    }
}

def shortest_route(graph, start, target):
    if start not in graph or target not in graph:
        return None
    q = deque([(start, [], [start])])
    seen = {start}
    while q:
        room, route, path = q.popleft()
        if room == target:
            return {"route": route, "path": path}
        for direction, nxt in graph.get(room, {}).items():
            if nxt not in seen:
                seen.add(nxt)
                q.append((nxt, route + [direction], path + [nxt]))
    return None

for target in ["Kitchen", "Cellar", "Troll Room", "East-West Passage", "Loud Room", "Up a Tree", "Attic"]:
    route = shortest_route(WORLD_MODEL["navigation_graph"], WORLD_MODEL["current_location"], target)
    if route:
        WORLD_MODEL["routes_from_current_location"][target] = route["route"]

print(json.dumps(WORLD_MODEL, indent=2))


In [ ]:
TEST_CASES = [
    {
        "id": "route_living_to_troll",
        "question": "You are in Living Room. To go from Living Room to Troll Room, what route should you follow?",
        "start": "Living Room",
        "target_location": "Troll Room",
    },
    {
        "id": "route_loud_to_kitchen",
        "question": "You are in Loud Room now. What route gets you back to Kitchen?",
        "start": "Loud Room",
        "target_location": "Kitchen",
    },
    {
        "id": "route_clearing_to_living",
        "question": "You are in Clearing. What route should you follow to reach Living Room?",
        "start": "Clearing",
        "target_location": "Living Room",
    },
    {
        "id": "object_egg_from_living",
        "question": "You are in Living Room and need the egg. Where is the egg, and what route should you follow to get there?",
        "start": "Living Room",
        "target_object": "egg",
    },
    {
        "id": "object_axe_from_kitchen",
        "question": "You are in Kitchen and need the axe. Where is the axe, and what route should you follow to get there?",
        "start": "Kitchen",
        "target_object": "axe",
    },
    {
        "id": "unknown_route_to_dam",
        "question": "You are in Living Room. What route should you follow to reach Dam?",
        "start": "Living Room",
        "target_location": "Dam",
    },
    {
        "id": "unknown_object_rope",
        "question": "You are in Living Room and need the rope. Where is the rope, and what route should you follow?",
        "start": "Living Room",
        "target_object": "rope",
    },
]

def expected_for_case(case):
    graph = WORLD_MODEL["navigation_graph"]
    start = case["start"]
    if "target_object" in case:
        obj = case["target_object"]
        loc = WORLD_MODEL["known_object_locations"].get(obj)
        if not loc:
            return {"known": False, "target_object": obj, "target_location": None, "route": None, "path": None}
        route = shortest_route(graph, start, loc)
        return {"known": bool(route), "target_object": obj, "target_location": loc, **(route or {"route": None, "path": None})}
    target = case["target_location"]
    route = shortest_route(graph, start, target)
    return {"known": bool(route), "target_location": target, **(route or {"route": None, "path": None})}

for case in TEST_CASES:
    print(case["id"], "=>", expected_for_case(case))


In [ ]:
SYSTEM_PROMPT = """You are a map-comprehension evaluator for a text adventure agent.
Use ONLY the provided JSON world model. Do not use outside knowledge about Zork.
If a requested room, object, or route is not present or not connected in the JSON, say it is unknown.
Return JSON only between |start| and |end|.

Required output schema:
{
  "known": true or false,
  "start": "room name",
  "target_location": "room name or null",
  "target_object": "object name or null",
  "route": ["direction", "direction"],
  "path": ["room", "room"],
  "reason": "one short sentence"
}

Rules:
- route is a list of direction commands only, such as ["down", "north"].
- path is a list of room names from start to target.
- If unknown, set known=false, route=[], path=[], and explain what is missing from the JSON.
"""

def build_user_prompt(case):
    return f"""World model JSON:
{json.dumps(WORLD_MODEL, indent=2)}

Question:
{case['question']}
"""

def ask_qwen(case):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(case)},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def parse_answer(raw):
    m = re.search(r"\|start\|\s*(.*?)\s*\|end\|", raw, re.S)
    body = m.group(1).strip() if m else raw.strip()
    try:
        return json.loads(body)
    except Exception as e:
        return {"parse_error": str(e), "raw_body": body}

def route_matches(answer, expected):
    if bool(answer.get("known")) != bool(expected.get("known")):
        return False
    if not expected.get("known"):
        return True
    return answer.get("route") == expected.get("route")

RESULTS = []
for case in TEST_CASES:
    print("\n" + "=" * 90)
    print(case["id"])
    print(case["question"])
    expected = expected_for_case(case)
    print("EXPECTED:", expected)
    raw = ask_qwen(case)
    parsed = parse_answer(raw)
    ok = route_matches(parsed, expected)
    print("RAW:\n", raw)
    print("PARSED:", parsed)
    print("PASS:", ok)
    RESULTS.append({"case": case, "expected": expected, "raw": raw, "parsed": parsed, "pass": ok})

print("\nSUMMARY")
print(sum(1 for r in RESULTS if r["pass"]), "/", len(RESULTS), "passed")
